In [4]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:

df = pd.read_csv("../data/raw/spotify_tracks_raw.csv")

In [ ]:
df = df.dropna()
df["genres"] = df["genres"].apply(eval)  # jeśli zapisane jako string
df = df[df["genres"].map(len) > 0]

In [ ]:
# ENCODING GENRES
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df["genres"])
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)

df_model = pd.concat([df, genre_df], axis=1)

In [ ]:
# FEATURES
feature_cols = ["popularity", "duration_sec", "explicit", "artist_popularity"]

X = pd.concat([
    df_model[feature_cols],
    genre_df
], axis=1)

In [ ]:
# SCALING
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# CLUSTERING
kmeans = KMeans(n_clusters=4, random_state=42)
df["cluster"] = kmeans.fit_predict(X_scaled)


In [ ]:
# ANALYSIS
df.groupby("cluster")["genres"].apply(lambda x: x.explode().value_counts().head(5))

In [ ]:
# VISUALIZATION
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:,0], X_pca[:,1], c=df["cluster"])
plt.show()